## Model Categories

#### Ultra Light: 0B-1B
1) https://huggingface.co/JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const_0.3 (0.6B)
2) https://huggingface.co/KingNish/Reasoning-0.5b (0.5B)
3) https://huggingface.co/oopere/Llama-3.2-1B-pruned-40pct (0.9B)
#### Small:  2B-3B
1) https://huggingface.co/Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1 (3B)
2) https://huggingface.co/google/gemma-2-2b-it?library=transformers (3B)
3) https://huggingface.co/SpaceYL/ECE_Poirot (2B)
#### Medium: 3B-7B
1) https://huggingface.co/OpenBuddy/openbuddy-zero-3b-v21.2-32k (5B)
2) https://huggingface.co/jpacifico/Distilucie-7B-Math-Instruct-DPO-v0.1 (7B)
3) https://huggingface.co/weathermanj/Menda-3b-Optim-100 (3B)
#### Large: 7B-10B
1) https://huggingface.co/grimjim/Magnolia-v3-Gemma2-8k-9B (8B)
2) https://huggingface.co/marcuscedricridia/cursa-o1-7b-v1.2-normalize-false?library=transformers
3) https://huggingface.co/marcuscedricridia/Abus-7B-Instruct (8B)
#### Massive: 10 or higher
1) [https://huggingface.co/MaziyarPanahi/calme-3.2-instruct-78b (78B)](https://huggingface.co/marcuscedricridia/Abus-7B-Instruct) (10B)
2) https://huggingface.co/arcee-ai/Virtuoso-Lite (10B)
3) https://huggingface.co/Aashraf995/Gemma-Evo-10B (10B)

## Prompt Types

1) Text generation
2) Classification
3) Summarization
4) Prediction
5) Extraction

## Evaluation Prompts

#### Text Generation
1) What are the legal moves for a knight in Chess?
2) What shoes would you recommend for a beginner runner?
3) What is the easiest sweet to bake?
4) What is the bar scene like in Champaign-Urbana?
5) What position should I play in basketball?

#### Classification
1) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: A group game of Axis & Allies.
2) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Hiking at Turkey Run State Park.
3) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Setting up the telescope on the main quad.
4) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Getting a group together for a sunday morning run.
5) Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Organizing a campus bar crawl.

#### Summarization
1) What are objectives of the game Catan?
2) Can you summarize the basics of Chess?
3) What are the main benefits of weightlifting?
4) Can you give me a quick summary of ingredients for baking a cake?
5) Give me a quick summary of the most popular activities for a college town.

#### Prediction
1) If I do the King's Pawn Opening, what move can I expect my opponent to do?
2) I like Super Smash Brothers, what other fighting games do you think I'd enjoy?
3) How fast can my group of 8 adults do a four-mile hike with 2,000 feet of elevation gain?
4) I am a 21-year-old male and run an 18:50 5k, what is my V02 max?
5) What activities will be the most popular in Champaign-Urbana tonight?

#### Extraction
1) Review all of my favorite activities (baking, basketball, and archery) and provide estimates for the price of each one.
2) Can you translate this username in my friendslist (闇風) into english?
3) Can you give me a random list of uniquely typed activties?
4) Can you show me which of my friends have been active the last 30 days? (jerry: 1, Larry: 0, Ryan: 0, Sophie: 1, Maggie: 1)
5) How many friends do I have? (Jerry, Larry, Ryan, Sophie, Maggie)

## Define Evaluation Metrics

Metric: Accuracy (Was the answer correct?)
Metric: Clarity (Was the output understandable?)
Metric: Instruction adherence (Did the model follow the instructions?)


## Benchmarking

In [22]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
"""
Benchmarking method, compiled from Week 6 scripts - script-3.ipynb
"""

def benchmarking(model_name, prompt, max_new_tokens=60):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    model.eval()
    start = time.time()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - start
    new_tokens = output.shape[1] - input_len
    text = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

    return {
        "model": model_name,
        "latency_sec": round(latency, 3),
        "new_tokens": new_tokens,
        "tokens_per_sec": round(new_tokens / latency, 2),
        "response": text
    }

In [ ]:

models = [ # ultra light
            "JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const",
          "KingNish/Reasoning-0.5b",
          "oopere/Llama-3.2-1B-pruned-40pct",

          # small
          "Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1",
          "google/gemma-2-2b-it",
           "SpaceYL/ECE_Poirot",

         # Medium
        "OpenBuddy/openbuddy-zero-3b-v21.2-32k",
        "Qwen/Qwen2.5-1.5B-Instruct",
        "weathermanj/Menda-3b-Optim-100",

        # Large

        "grimjim/Magnolia-v3-Gemma2-8k-9B",
        "marcuscedricridia/cursa-o1-7b-v1.2-normalize-false",
        "marcuscedricridia/Abus-7B-Instruct"
]


prompts = [

    # Text generation

    ("Generate", "What are the legal moves for a knight in Chess?"),
    ("Generate", "What shoes would you recommend for a beginner runner?"),
    ("Generate", "What is the easiest sweet to bake?"),
    ("Generate", "What is the bar scene like in Champaign-Urbana?"),
    ("Generate", "What position should I play in basketball?"),

    # Classification

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: A group game of Axis & Allies."),

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Hiking at Turkey Run State Park."),

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Setting up the telescope on the main quad."),

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Getting a group together for a sunday morning run."),

     ("Classification", "Classify the activity type out of Board Games, Fitness, Outdoors, Culinary, Leisure, and Social: Organizing a campus bar crawl."),

    # Summarization

     ("Summarization", "What are objectives of the game Catan?"),
     ("Summarization", "Can you summarize the basics of Chess?"),
     ("Summarization", "What are the main benefits of weightlifting?"),
     ("Summarization", "Can you give me a quick summary of ingredients for baking a cake?"),
     ("Summarization", "Give me a quick summary of the most popular activities for a college town."),

    # Prediction

    ("Prediction", "If I do the King's Pawn Opening, what move can I expect my opponent to do?"),
    ("Prediction", "I like Super Smash Brothers, what other fighting games do you think I'd enjoy?"),
    ("Prediction", "How fast can my group of 8 adults do a four-mile hike with 2,000 feet of elevation gain?"),
    ("Prediction", "I am a 21-year-old male and run an 18:50 5k, what is my V02 max?"),
    ("Prediction", "What activities will be the most popular in Champaign-Urbana tonight?"),

    # Extraction

     ("Extraction", "Review all of my favorite activities (baking, basketball, and archery) and provide estimates for the price of each one."),

     ("Extraction", "Can you translate this username in my friendslist (闇風) into english?"),

     ("Extraction", "Can you give me a random list of uniquely typed activties?"),

     ("Extraction", "Can you show me which of my friends have been active the last 30 days? (jerry: 1, Larry: 0, Ryan: 0, Sophie: 1, Maggie: 1)."),

     ("Extraction", "How many friends do I have? - Jerry, Larry, Ryan, Sophie, Maggie.")
]

for model in models:
    for prompt, p in prompts:
        result = benchmarking(model, p)
        print(model, prompt, result["tokens_per_sec"], result["latency_sec"])



JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 11.96 5.017
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 12.72 4.718
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 12.61 4.757
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 9.08 6.605
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Generate 9.09 6.599
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 10.01 5.995
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 12.84 4.674
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 9.29 1.4
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 9.35 3.959
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Classification 9.12 6.58
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 7.49 8.009
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 12.48 4.808
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 13.65 4.396
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 7.83 7.663
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Summarization 13.72 4

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1 Generate 2.66 22.598


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1 Generate 3.42 17.554


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

weathermanj/Menda-3b-Optim-100 Extraction 2.53 23.743


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

weathermanj/Menda-3b-Optim-100 Extraction 2.83 12.726


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Note: my machine heavily struggles to run models larger than 10B parameters. This section does not include the massive models because it would take hours to run them. It took over an hour to run the previous cells.

# Analysis

1) It appears that text generation was the best category.
2) A larger model doesn't necessarily indicate better performance. As they grow, the time for response increases. This wouldn't be practical for our app, users expect fast response.
3) Ultimately, the KingNish/Reasoning-0.5b model has the best balance between speed/quality.